In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('creditcard.csv')
print(df.shape)
df.head()


In [ ]:
#cell 2 Missing value Check
print ("Missing values per column:")
print(df.isnull().sum())
print("\nTotal missing values:" , df.isnull().sum().sum())

In [ ]:
# class distribution
print ("class distribution:")
print (df['Class'].value_counts())
print("\nFraud Percentage:")
print(df['Class'].value_counts(normalize=True)*100)


In [ ]:
# Cell 4 -Basic statistics
df.describe()

In [ ]:
# Cell 5- Legit Vs Fraud Detection
df.groupby('Class')['Amount'].describe()

In [ ]:
# Cell 6 - Correlation Heatmap
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True) , cmap= 'coolwarm' , center=0 , annot=False)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:
# Cell 7 - Amount and Time Distribution
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Amount distribution
sns.histplot(df['Amount'], bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount')

# Time distribution
sns.histplot(df['Time'], bins=50, ax=axes[1], color='coral')
axes[1].set_title('Transaction Time Distribution')
axes[1].set_xlabel('Time (seconds)')

plt.tight_layout()
plt.show()


In [ ]:
# Cell 8 - Fraud vs Legit Amount Comparison
plt.figure(figsize=(10,5))
sns.boxplot(x='Class', y='Amount', data=df)
plt.title('Fraud vs Legit - Amount Comparison')
plt.xticks([0,1], ['Legit', 'Fraud'])
plt.tight_layout()
plt.show()

## Modelling has moved out of this notebook

Cells 9-16 (scaling, train/test split, SMOTE, XGBoost training, evaluation,
SHAP, threshold tuning, model save) were removed. They are replaced by:

| was | now |
|---|---|
| cell 9 — StandardScaler on Amount/Time | `features.py` — stateless `engineer()`, `Time` -> `hour_of_day`, nothing scaled |
| cells 10-12 — split, SMOTE, fit | `train_pipeline.py` |
| cell 13 — classification report / confusion matrix | `train_pipeline.py` (plus full threshold sweep) |
| cell 14 — SHAP | `explain.py` — held-out rows, version-stamped, refuses to load stale |
| cell 15 — five hardcoded thresholds | `train_pipeline.py` threshold sweep -> `artifacts/threshold_sweep_v2.0.0.csv` |
| cell 16 — pickle dump + best_threshold.txt | `train_pipeline.py` bundle (`fraudshield_bundle_latest.pkl`) |

This notebook keeps what a notebook is actually good for: exploratory data
analysis (cells 1-8) and the RAG layer (cells 17-22).

Run before using the cells below:

```bash
python train_pipeline.py
python serving.py          # both invariants must PASS
python explain.py          # regenerates SHAP against the v2 bundle
```


In [ ]:
# Cell 17 - RAG Setup with LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline
import pandas as pd
import numpy as np

print("All imports successful!")


In [ ]:
# Cell 18 - Create Knowledge Base
fraud_knowledge = """
FRAUD DETECTION RULES AND EXPLANATIONS:

1. HIGH AMOUNT TRANSACTIONS:
Transactions above $5000 are considered high risk.
Large amounts significantly increase fraud probability.
SHAP value for Amount feature is typically high for fraud cases.

2. TIME BASED PATTERNS:
Transactions between 2AM-4AM (low time values) are suspicious.
Unusual timing indicates potential fraud activity.

3. V1-V28 PCA FEATURES:
V1, V2, V3, V4, V10, V11, V12, V14, V17 are most important features.
Negative values in V14 strongly indicate fraud.
Negative values in V17 strongly indicate fraud.
Extreme values in V1 and V3 suggest suspicious activity.

4. THRESHOLD EXPLANATION:
Our model uses 0.7 threshold for fraud classification.
This means 70% confidence required to flag as fraud.
Higher threshold = fewer false positives but may miss some fraud.

5. PRECISION AND RECALL:
Current model precision: 51% - of flagged transactions, 51% are actual fraud.
Current model recall: 87% - model catches 87% of all actual fraud cases.
We prioritize recall to minimize missed fraud cases.
"""

print("✅ Knowledge base created!")
print(f"Total characters: {len(fraud_knowledge)}")


In [ ]:
# Cell 19 - Split Knowledge Base into Chunks
text_splitter = RecursiveCharacterTextSplitter(
chunk_size=200,
chunk_overlap=20
)

chunks = text_splitter.create_documents([fraud_knowledge])

print(f"✅ Text split into {len(chunks)} chunks!")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}: {chunk.page_content[:100]}...")


In [ ]:
# Cell 20 - Create FAISS Vector Store
embeddings = HuggingFaceEmbeddings(
model_name="all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embeddings)

print("✅ Vector store created!")
print(f"Total vectors stored: {vectorstore.index.ntotal}")

In [ ]:
# Cell 21 - SHAP-aware RAG query (v2)
#
# Was: read `shap_values` / `X_test_sample` as mid-notebook globals, which is
# why the pickles went stale without anything complaining. Now it goes through
# explain.load(), which asserts the SHAP artifact was built against the model
# currently in the bundle and raises if it wasn't.
#
# Also fixes the Amount bug: `tx["Amount", "N/A"] if "Amount" in tx.index`
# was always False under v1 (Amount had been dropped for Amount_scaled), so
# every RAG context ever generated said "Amount: $0.00". Under v2, Amount
# survives feature engineering and top_features() reports it in euros.

from explain import load as load_shap, top_features

shap_payload = load_shap()
print(f"SHAP artifact: model {shap_payload['model_version']}, "
      f"{len(shap_payload['X_sample'])} rows from {shap_payload['source']}")


def ask_fraudshield(question, transaction_idx=None):
    """Retrieval + SHAP context. Note this returns a *prompt*, not an answer --
    there is no generation step yet, so this is retrieval, not RAG. Wiring the
    LLM in is a separate change."""
    docs = vectorstore.similarity_search(question, k=2)
    rag_context = "\n".join(d.page_content for d in docs)

    if transaction_idx is None:
        full_context = rag_context
    else:
        full_context = (
            top_features(transaction_idx, k=3, payload=shap_payload)
            + "\n\nGENERAL RULES:\n"
            + rag_context
        )
    return f"QUESTION: {question}\n\nCONTEXT:\n{full_context}"


print(ask_fraudshield("Why was this transaction flagged as fraud?", transaction_idx=0))


In [ ]:
# Save FAISS index to disk
vectorstore.save_local("faiss_index")
print("FAISS index saved")